# CUDA Lab
 Orynbay Ali


## Section 1: Hardware telemetry

In [ ]:
!nvidia-smi

Thu Oct  8 07:17:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from numba import cuda
import os

device = cuda.get_current_device()
name = device.name.decode('utf-8') if isinstance(device.name, bytes) else device.name
print(f"Device Name: {name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


## Section 2: Crossover experiment

In [ ]:
import numpy as np
import time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [ ]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [ ]:
sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
results = {}
print(f"{'N':>12} | {'CPU ms':>12} | {'GPU Total ms':>13} | {'GPU Kernel ms':>14} | Winner")
print("-" * 75)
for N in sizes:
    cpu, tot, ker = benchmark_run(N)
    results[N] = (cpu, tot, ker)
    if cpu != cpu:  # nan
        winner = "n/a (CPU skipped)"
    else:
        winner = "GPU" if tot < cpu else "CPU"
    print(f"{N:>12,} | {cpu:>12.4f} | {tot:>13.4f} | {ker:>14.4f} | {winner}")

CROSSOVER_N = None
for N in sizes:
    cpu, tot, ker = results[N]
    if cpu == cpu and tot < cpu:
        CROSSOVER_N = N
        break
print("\nCrossover point N* =", CROSSOVER_N)

           N |       CPU ms |  GPU Total ms |  GPU Kernel ms | Winner
---------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


         100 |       0.0529 |        1.3855 |         0.0928 | CPU
       1,000 |       0.2799 |        0.6426 |         0.0845 | CPU
      10,000 |       2.7563 |        0.5442 |         0.0875 | GPU
     100,000 |      27.3833 |        0.7482 |         0.0717 | GPU
   1,000,000 |     268.4608 |        4.9282 |         0.1056 | GPU
   5,000,000 |          nan |       12.3829 |         0.2565 | n/a (CPU skipped)
  10,000,000 |          nan |       25.5676 |         0.4743 | n/a (CPU skipped)

Crossover point N* = 10000


## Section 3: Thread geometry

### Task 3.1

In [ ]:
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [ ]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
total = blocks_per_grid * threads_per_block

log_b = cuda.device_array(total, dtype=np.int32)
log_t = cuda.device_array(total, dtype=np.int32)
log_g = cuda.device_array(total, dtype=np.int32)
log_a = cuda.device_array(total, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

hb, ht, hg, ha = log_b.copy_to_host(), log_t.copy_to_host(), log_g.copy_to_host(), log_a.copy_to_host()
print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):
    status = "ACTIVE" if ha[i] == 1 else "IDLE (GUARDED)"
    print(f"{hg[i]:<10} | {hb[i]:<10} | {ht[i]:<10} | {status:<10}")

print("\nTotal blocks:", blocks_per_grid)
print("Total threads:", total)
print("Active threads:", int(ha.sum()))
print("Idle threads:", total - int(ha.sum()))

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)

Total blocks: 5
Total threads: 40
Active threads: 37
Idle threads: 3


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


## Section 4: Race condition

In [ ]:
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
d_ones = cuda.to_device(np.ones(N, dtype=np.float32))
tpb = 256
bpg = (N + tpb - 1) // tpb

for k in (naive_sum_kernel, atomic_sum_kernel):
    k[bpg, tpb](d_ones, cuda.to_device(np.zeros(1, dtype=np.float32)))
cuda.synchronize()

print(f"{'Trial':<8}{'Target':>10}{'Naive output':>16}{'Lost':>12}")
for t in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[bpg, tpb](d_ones, d_total)
    cuda.synchronize()
    val = float(d_total.copy_to_host()[0])
    print(f"Trial {t:<2}{50000.0:>10.1f}{val:>16.1f}{50000.0 - val:>12.1f}")

Trial       Target    Naive output        Lost
Trial 1    50000.0             2.0     49998.0
Trial 2    50000.0             2.0     49998.0
Trial 3    50000.0             2.0     49998.0
Trial 4    50000.0             2.0     49998.0
Trial 5    50000.0             2.0     49998.0


In [ ]:
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[bpg, tpb](d_ones, d_total)
cuda.synchronize()
atomic_val = float(d_total.copy_to_host()[0])
print("Atomic sum:", atomic_val, "| equals 50000.0?", atomic_val == 50000.0)

def time_kernel(kernel, reps=20):
    ts = []
    for _ in range(reps):
        d_t = cuda.to_device(np.zeros(1, dtype=np.float32))
        cuda.synchronize()
        t0 = time.perf_counter()
        kernel[bpg, tpb](d_ones, d_t)
        cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000.0)
    return sum(ts) / len(ts)

naive_ms = time_kernel(naive_sum_kernel)
atomic_ms = time_kernel(atomic_sum_kernel)
print(f"Naive kernel:  {naive_ms:.4f} ms")
print(f"Atomic kernel: {atomic_ms:.4f} ms")

Atomic sum: 50000.0 | equals 50000.0? True
Naive kernel:  0.1629 ms
Atomic kernel: 0.3214 ms


## Section 5: 2D grid

In [ ]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()

print(f"Generated 2D matrix shape: {result_canvas.shape}. Center value: {result_canvas[512, 512]:.3f}")
print(f"Blocks in X: {blocks_2d[0]} | Blocks in Y: {blocks_2d[1]} | Total: {blocks_2d[0]*blocks_2d[1]}")
print(f"Pixel (512,512): {result_canvas[512,512]:.4f}")
print(f"Pixel (0,0):     {result_canvas[0,0]:.4f}")

Generated 2D matrix shape: (1024, 1024). Center value: 1.000
Blocks in X: 64 | Blocks in Y: 64 | Total: 4096
Pixel (512,512): 1.0000
Pixel (0,0):     0.0000


## Section 6: Verification checksum


In [ ]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    gpu_name = cuda.get_current_device().name
    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode('utf-8')
    h.update(gpu_name)
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

STUDENT_ID = "230103160"
generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 718628F16D347147


'718628F16D347147'